# Lesson 4.5: Building RAG from Scratch

**The 100-line pattern that powers every AI product you've ever used.**

45-60 min. Code-heavy. No frameworks -- just `numpy` and `openai`.

## Architect Level

- [ ] Junior (1-2 yrs) -- implement and run
- [ ] Mid (3-5 yrs) -- extend with monitoring
- [ ] Senior (6-10 yrs) -- add multi-tenancy
- [ ] Staff (10+ yrs) -- design the SLA, capacity model, incident playbook

---

## Concept (5 min)

**RAG (Retrieval-Augmented Generation)** is three steps:

1. **Embed** your documents into vectors (numerical representations of meaning).
2. **Retrieve** the most similar documents to a user's question (by vector distance).
3. **Generate** an answer using the LLM with the retrieved documents as context.

Why does it work? Because **embeddings capture meaning**. The vector for "king" minus "man" plus "woman" is close to "queen". So a search for "how do I cancel my subscription" can find the doc titled "Termination Policy".

Architect's mental model:
- **Embeddings are lossy.** They capture meaning, not exact wording. Use keyword search alongside.
- **Chunking is the secret sauce.** Most RAG failures are chunking failures.
- **Context window is your budget.** You can fit ~3-5 chunks in a 4K-token context. Choose wisely.

## Setup

Install dependencies and set your API key.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key
# !pip install openai numpy

## Starter Code (TODOs)

Implement these 4 functions.

In [ ]:
import numpy as np
from openai import OpenAI

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
TOP_K = 3

def get_embedding(client, text):
    """TODO: Return the embedding vector for the text."""
    pass

def cosine_similarity(a, b):
    """TODO: Return the cosine similarity between vectors a and b."""
    pass

def retrieve(client, query, doc_embeddings, top_k=TOP_K):
    """TODO: Embed query, score all docs, return top_k (text, score) pairs."""
    pass

def generate_answer(client, question, context_docs):
    """TODO: Call LLM with question + context. Return the answer string."""
    pass

## Solution -- Complete Version

In [ ]:
def get_embedding(client, text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(client, query, doc_embeddings, top_k=TOP_K):
    q_emb = get_embedding(client, query)
    scored = [(doc, cosine_similarity(q_emb, emb)) for doc, emb in doc_embeddings]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

def generate_answer(client, question, context_docs):
    context_str = "\n".join(f"- {doc}" for doc, _ in context_docs)
    system_prompt = (
        "You are a helpful customer support assistant. "
        "Answer using ONLY the context below. If not in context, say 'I don't know.'\n\n"
        f"Context:\n{context_str}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

## Demo: Customer Support RAG

In [ ]:
DOCUMENTS = [
    "Our company offers a 30-day money-back guarantee on all products.",
    "Shipping is free for orders over $50. Standard shipping takes 3-5 business days.",
    "To cancel a subscription, go to Account Settings > Subscriptions and click Cancel.",
    "Refund requests are processed within 5-7 business days after we receive the item.",
    "We accept Visa, Mastercard, American Express, and PayPal.",
    "Customer support is available 24/7 via email and live chat.",
    "Bulk orders of $500+ receive a 10% discount automatically at checkout.",
    "Returns must be in original packaging with all tags attached.",
    "Account deletion is permanent and removes all associated data within 30 days.",
    "Two-factor authentication is available and recommended for all accounts.",
    "Premium members get early access to sales and exclusive product launches.",
    "International shipping rates are calculated at checkout based on destination.",
]

client = OpenAI()
print(f"Embedding {len(DOCUMENTS)} documents...")
doc_embeddings = [(doc, get_embedding(client, doc)) for doc in DOCUMENTS]
print(f"Done.\n")

In [ ]:
questions = [
    "How do I cancel my subscription?",
    "What is your refund policy?",
    "Do you ship internationally?",
    "What payment methods do you accept?",
]

for q in questions:
    print(f"\n  Q: {q}")
    retrieved = retrieve(client, q, doc_embeddings, top_k=3)
    answer = generate_answer(client, q, retrieved)
    print(f"  A: {answer}")
    print(f"  Retrieved (with scores):")
    for doc, score in retrieved:
        print(f"    [{score:.3f}] {doc[:70]}")

## Architect Notes

### Trade-offs

| Approach       | Pros                          | Cons                                      | Pick when                        |
|----------------|-------------------------------|-------------------------------------------|----------------------------------|
| RAG (this lab) | Always up-to-date, low cost  | Limited by retrieval quality              | Most cases (docs, Q&A, support)  |
| Fine-tuning    | Learns domain knowledge       | Expensive, slow to update                 | Repeated patterns, specific style|
| Long context   | No retrieval needed           | Expensive ($5/query at 200K), slow        | Small doc collections (<100)     |
| Hybrid         | Best of both                  | More moving parts                         | Production-grade Q&A             |

### Capacity Model

| Volume      | Embeddings latency        | Retrieval latency | LLM latency | Total p99 |
|-------------|---------------------------|-------------------|-------------|----------|
| 100 docs    | 2-3 sec (one-time)        | <1 ms             | 1-2 sec     | ~3 sec   |
| 10K docs    | 5-10 min (one-time)       | <10 ms            | 1-2 sec     | ~2 sec   |
| 1M docs     | 8-15 hours (one-time)     | 50-200 ms         | 1-2 sec     | ~2.5 sec |
| 100M docs   | Days (one-time, parallel) | 100-500 ms        | 1-2 sec     | ~3 sec   |

For <100K docs, in-memory numpy is fine. For 1M+, use a vector DB (Pinecone, Weaviate, Qdrant) with HNSW.

### Cost Model (per 1K queries, GPT-4o-mini)

| Component                        | Cost     |
|----------------------------------|----------|
| Embeddings (50 tokens × 1K)      | $0.01    |
| Vector search                    | $0       |
| LLM (1K input + 200 output × 1K) | $0.27    |
| **Total per 1K queries**         | **$0.28**|

At 1M queries/day: $280/day = $8,400/month in LLM cost alone.

### When NOT to use RAG

- Real-time data (stock prices, weather) -- use tools/functions
- Exact-match lookups (order by ID) -- use SQL
- Tasks where the LLM already knows the answer -- RAG adds cost with no quality gain

### Production Checklist

- [ ] Embedding model is pinned (don't switch without re-embedding)
- [ ] Documents chunked (200-1000 tokens, 10-20% overlap)
- [ ] Each chunk has metadata: source, timestamp, author, doc_id
- [ ] Hybrid search: vector + keyword (BM25)
- [ ] Top-k bounded (3-10 chunks)
- [ ] LLM context sized correctly (don't send 100K if 2K suffices)
- [ ] Eval set exists (50+ Q&A pairs) and is run on every change
- [ ] RAGAS measures faithfulness, relevance, context precision
- [ ] Caching layer for repeated queries
- [ ] Rate limits per user
- [ ] "I don't know" response when retrieval confidence is low

## Reflect (10 min)

1. What did I build?  A RAG system. What surprised you about how easy the core pattern is?
2. What was hard?  The cosine similarity? The prompt for the generation step?
3. What would I change at 10x scale?  If you had 1M documents instead of 10, what breaks?
4. What's tomorrow's lab?  Lesson 4.6: LangChain RAG. Same thing in 30 lines using a framework.

---

**References**

- [Codebook section 4.1](../../workbooks/ai-engineer-codebook.md#41-simple-rag-no-frameworks) -- same in 50 lines
- [Codebook section 4.2](../../workbooks/ai-engineer-codebook.md#42-langchain-rag) -- same in LangChain
- [Codebook section 4.5](../../workbooks/ai-engineer-codebook.md#45-rag-evaluation-with-ragas) -- eval with RAGAS
- [Lesson script for 3.2](../../scripts/lesson-3-2-rag-script.md) -- full 28-min video script
- [RAGAS docs](https://docs.ragas.io/)
- [Pinecone learning center](https://www.pinecone.io/learn/)